# Lorenz-63 demo (unforced vs forced)

This notebook introduces the classic Lorenz-63 system and shows how to run it within the Paleobeasts model API.
We compare an unforced run against a simple periodic forcing applied to `dx/dt`.

**Why Lorenz-63?** The system is a canonical example of deterministic chaos. It is low-dimensional (3 variables),
but it exhibits sensitive dependence on initial conditions and produces the well-known butterfly attractor.


## How the Paleobeasts model API works

1. Create a `Forcing` object (even if it is constant zero).
2. Instantiate the model with that forcing.
3. Call `integrate(...)` to solve over a time span.
4. Optionally reframe onto an evenly-spaced time axis with `reframe_time_axis(...)`.

This keeps the runtime efficiency of adaptive methods like RK45 while giving you a clean grid for downstream analysis.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import paleobeasts as pb
from paleobeasts.signal_models import Lorenz63


In [ ]:
# Common integration setup
t_span = (0.0, 40.0)
y0 = [1.0, 1.0, 1.0]
t_eval = pb.utils.define_t_eval(t_span, delta_t=0.01)


## Unforced Lorenz-63

We pass a constant-zero forcing so the system evolves only under its intrinsic dynamics.
After integrating with RK45, we reframe onto an even time grid so plots and statistics are straightforward.

In [ ]:
unforced = pb.core.Forcing(lambda t: 0.0)
lorenz_unforced = Lorenz63(forcing=unforced)
lorenz_unforced.integrate(t_span=t_span, y0=y0, method='RK45')
lorenz_unforced.reframe_time_axis(t_eval, update_state=True)

x_u = lorenz_unforced.state_variables['x']
y_u = lorenz_unforced.state_variables['y']
z_u = lorenz_unforced.state_variables['z']
t_u = lorenz_unforced.time


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(t_u, x_u, label='x')
ax[0].plot(t_u, y_u, label='y')
ax[0].plot(t_u, z_u, label='z')
ax[0].set_title('Time series (unforced)')
ax[0].set_xlabel('t')
ax[0].legend()

ax[1].plot(x_u, z_u, linewidth=0.6)
ax[1].set_title('Phase portrait (x vs z)')
ax[1].set_xlabel('x')
ax[1].set_ylabel('z')

plt.tight_layout()


## Forced Lorenz-63

Here we apply a sinusoidal forcing to `dx/dt`. This is a simple way to represent external modulation,
which can be useful for sensitivity tests or for coupling the system to a driver.

In [ ]:
def forcing_func(t):
    return 2.0 * np.sin(2 * np.pi * t / 8.0)

forced = pb.core.Forcing(forcing_func)
lorenz_forced = Lorenz63(forcing=forced)
lorenz_forced.integrate(t_span=t_span, y0=y0, method='RK45')
lorenz_forced.reframe_time_axis(t_eval, update_state=True)

x_f = lorenz_forced.state_variables['x']
y_f = lorenz_forced.state_variables['y']
z_f = lorenz_forced.state_variables['z']
t_f = lorenz_forced.time


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(t_f, x_f, label='x')
ax[0].plot(t_f, y_f, label='y')
ax[0].plot(t_f, z_f, label='z')
ax[0].set_title('Time series (forced)')
ax[0].set_xlabel('t')
ax[0].legend()

ax[1].plot(x_f, z_f, linewidth=0.6)
ax[1].set_title('Phase portrait (x vs z)')
ax[1].set_xlabel('x')
ax[1].set_ylabel('z')

plt.tight_layout()
